In [ ]:
# allows update of external libraries without need to reload package
%load_ext autoreload
%autoreload 2

# Enjoy synthetic 2D fields

## Summary

Experiment employing the synthetic fields.

## Results

### Key results

- Helpful functionality:
  - Equip synthetic fields w/ CRS using an additional function `phoibe.synthetic_data.fields.make_field_rio`.

### Details



In [ ]:
import logging

import cartopy.crs as ccrs
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import xarray

import phoibe

LOGGER = logging.getLogger("phoibe.geography.crs.reproject")
LOGGER.setLevel(level=logging.DEBUG)


LAND_CMAP = matplotlib.colors.LinearSegmentedColormap.from_list(
    "land_cmap", [(0.00, "#2e7d32"), (0.30, "#66bb6a"), (0.55, "#a1887f"), (0.75, "#9e9e9e"), (1.00, "#ffffff")]
)

## Plotting Helper

Required as long as plotting awaits packaging.

In [ ]:
def _get_value_ranges(da: xarray.DataArray, pct_lower=1, pct_upper=99) -> tuple[float, float]:
    vmin = float(np.nanpercentile(da, pct_lower))
    vmax = float(np.nanpercentile(da, pct_upper))
    return vmin, vmax


def plot_geodata(da: xarray.DataArray, cmap=LAND_CMAP):
    """Plot the elevation map in colours."""
    if hasattr(da, "rio"):
        epsg_code = int(da.rio.crs.to_authority()[1])
        if epsg_code == 4326:
            plot_crs = ccrs.PlateCarree()
        elif epsg_code is not None:
            plot_crs = ccrs.epsg(epsg_code)
        else:
            plot_crs = None
    else:
        plot_crs = None
    # plot_crs = None
    figure = plt.figure()
    ax = figure.add_subplot(1, 1, 1, projection=plot_crs)

    da_clean = da.where(da != da.rio.nodata) if da.rio.nodata is not None else da
    vmin, vmax = _get_value_ranges(da=da_clean, pct_lower=1, pct_upper=99)
    cnorm = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax)

    mesh = ax.pcolormesh(da["x"], da["y"], da_clean, cmap=cmap, shading="auto", norm=cnorm, transform=plot_crs)

    ax.grid(visible=True)
    ax.gridlines(draw_labels=True)
    ax.set_title(f"CRS: {da.rio.crs.to_authority()}")

    cbar = plt.colorbar(mesh, ax=ax, orientation="vertical", shrink=0.7)
    cbar.set_label("Elevation [m]")
    plt.tight_layout()
    return figure, ax

## Create examplary field

In [ ]:
WIDTH, HEIGHT = 30, 20
eggbox = phoibe.synthetic_data.fields.make_eggbox_field(nx=WIDTH, ny=HEIGHT, dx=0.1, dy=0.1, freq_x=3, freq_y=3)
bounds = 8, 48, 9, 49

eggbox = phoibe.synthetic_data.fields.make_field_rio(da=eggbox, bounds=bounds, crs=4326, dtype="float", nodata=np.nan)

plot_geodata(eggbox)

## Reproject

In [ ]:
eggbox_utm, summary = phoibe.geography.crs.reproject_rasterdata(da=eggbox, crs_to=32632, resolution=30)
# da = eggbox_utm.copy()
# da = da.rio.write_crs(None)
plot_geodata(da=eggbox_utm)
eggbox_utm_gcs, summary = phoibe.geography.crs.reproject_rasterdata(da=eggbox_utm, crs_to=4326, resolution=0.1)
plot_geodata(da=eggbox_utm_gcs)
display(summary)

In [ ]:
display(eggbox.x)
display(eggbox_utm_gcs.x)

In [ ]:
def print_min_mean_max(da: xarray.DataArray, coord: str):
    values = getattr(da, coord)
    print(float(values.min()), float(values.mean()), float(values.max()), len(values))


print_min_mean_max(eggbox, "x")
print_min_mean_max(eggbox, "y")

print_min_mean_max(eggbox_utm, "x")
print_min_mean_max(eggbox_utm, "y")

print_min_mean_max(eggbox_utm_gcs, "x")
print_min_mean_max(eggbox_utm_gcs, "y")